In [2]:
print("ok all")

ok all


In [3]:
# !uv pip install llama-cloud

In [4]:
import os
import io
import re
import json
from pathlib import Path
from typing import Any

import pandas as pd
import requests
from dotenv import load_dotenv

from llama_cloud import LlamaCloud

In [5]:
# ============================================================
# 1. File Paths
# ============================================================

PDF_PATH = Path(
    r"F:\Full_Stack_GenerativeAI_And_Agentic_AI\RAG\Data_parsing\data\complex_rag_parsing_sample_with_sunny_image.pdf"
)

OUTPUT_DIR = Path(
    r"F:\Full_Stack_GenerativeAI_And_Agentic_AI\RAG\Data_parsing\data"
    r"\llamaparse_output"
)

IMAGE_DIR = OUTPUT_DIR / "extracted_images"
SCREENSHOT_DIR = IMAGE_DIR / "screenshots"
EMBEDDED_IMAGE_DIR = IMAGE_DIR / "embedded"
LAYOUT_IMAGE_DIR = IMAGE_DIR / "layout"
OTHER_IMAGE_DIR = IMAGE_DIR / "other"

TABLE_DIR = OUTPUT_DIR / "extracted_tables"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
SCREENSHOT_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDED_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
LAYOUT_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
OTHER_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF not found: {PDF_PATH}")

print("PDF found:", PDF_PATH)
print("Output directory:", OUTPUT_DIR)

PDF found: F:\Full_Stack_GenerativeAI_And_Agentic_AI\RAG\Data_parsing\data\complex_rag_parsing_sample_with_sunny_image.pdf
Output directory: F:\Full_Stack_GenerativeAI_And_Agentic_AI\RAG\Data_parsing\data\llamaparse_output


In [6]:
load_dotenv()

LLAMA_CLOUD_API_KEY = os.getenv("LLAMA_CLOUD_API_KEY")

if not LLAMA_CLOUD_API_KEY:
    raise ValueError(
        "LLAMA_CLOUD_API_KEY was not found.\n"
        "Create a .env file and add:\n"
        "LLAMA_CLOUD_API_KEY=llx-your-api-key"
    )

client = LlamaCloud(
    api_key=LLAMA_CLOUD_API_KEY
)

print("LlamaCloud client initialized successfully.")

LlamaCloud client initialized successfully.


In [7]:
# ============================================================
# 3. Helper Functions
# ============================================================

def safe_filename(filename: str) -> str:
    """
    Removes unsafe characters from a filename.
    """
    filename = Path(filename).name
    return re.sub(r'[<>:"/\\|?*]', "_", filename)


def object_to_dict(obj: Any) -> Any:
    """
    Safely converts an SDK/Pydantic object into a dictionary.
    """
    if obj is None:
        return None

    if isinstance(obj, dict):
        return {
            key: object_to_dict(value)
            for key, value in obj.items()
        }

    if isinstance(obj, (list, tuple)):
        return [object_to_dict(value) for value in obj]

    if hasattr(obj, "to_dict"):
        return object_to_dict(obj.to_dict())

    if hasattr(obj, "model_dump"):
        return object_to_dict(obj.model_dump(mode="json"))

    if hasattr(obj, "__dict__"):
        return {
            key: object_to_dict(value)
            for key, value in vars(obj).items()
            if not key.startswith("_")
        }

    return obj

    
def download_file(
    url: str,
    output_path: Path,
    timeout: int = 120
) -> bool:
    """
    Downloads a file from a presigned URL.
    """
    try:
        response = requests.get(
            url,
            timeout=timeout,
            stream=True
        )

        response.raise_for_status()

        output_path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        with output_path.open("wb") as file:
            for chunk in response.iter_content(
                chunk_size=1024 * 1024
            ):
                if chunk:
                    file.write(chunk)

        return True

    except Exception as error:
        print(
            f"Download failed for {output_path.name}: "
            f"{type(error).__name__}: {error}"
        )
        return False

def get_image_output_directory(category: str | None) -> Path:
    """
    Selects the correct folder according to image category.
    """
    category = (category or "other").lower()

    if category == "screenshot":
        return SCREENSHOT_DIR

    if category == "embedded":
        return EMBEDDED_IMAGE_DIR

    if category == "layout":
        return LAYOUT_IMAGE_DIR

    return OTHER_IMAGE_DIR

In [8]:
print("\nUploading PDF to LlamaCloud...")

uploaded_file = client.files.create(
    file=PDF_PATH,
    purpose="parse"
)

print("Upload completed.")
print("Uploaded file ID:", uploaded_file.id)


Uploading PDF to LlamaCloud...
Upload completed.
Uploaded file ID: 41c535a4-9cc3-45ce-9627-df60d8785739


In [9]:
# ============================================================
# 5. Parse PDF
# ============================================================

print("\nParsing PDF with LlamaParse...")

result = client.parsing.parse(
    file_id=uploaded_file.id,

    # Suitable for complex documents containing
    # tables, images, scans and mixed layouts.
    tier="agentic",

    version="latest",

    output_options={
        # Preserve tables in Markdown output.
        "markdown": {
            "tables": {
                "output_tables_as_markdown": True
            }
        },

        # Generate an Excel workbook containing tables.
        "tables_as_spreadsheet": {
            "enable": True
        },

        # Save complete page screenshots,
        # original embedded images and detected layout regions.
        "images_to_save": [
            "screenshot",
            "embedded",
            "layout"
        ],
    },

    processing_options={
        # OCR language configuration.
        "ocr_parameters": {
            "languages": ["en"]
        }
    },

    # Controls which results are returned.
    expand=[
        "text",
        "text_full",
        "markdown",
        "markdown_full",
        "items",
        "metadata",
        "job_metadata",
        "images_content_metadata",
        "xlsx_content_metadata",
    ],

    verbose=True,
    timeout=600,
)

print("\nParsing completed.")
print("Job ID:", result.job.id)
print("Job status:", result.job.status)


Parsing PDF with LlamaParse...

Parsing completed.
Job ID: pjb-m9y5gsn4euuqwtwzd3713litggsg
Job status: COMPLETED


In [10]:
raw_result = object_to_dict(result)

raw_result_path = OUTPUT_DIR / "llamaparse_raw_result.json"

with raw_result_path.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        raw_result,
        file,
        indent=2,
        ensure_ascii=False,
        default=str
    )

print("Raw JSON saved:", raw_result_path)

Raw JSON saved: F:\Full_Stack_GenerativeAI_And_Agentic_AI\RAG\Data_parsing\data\llamaparse_output\llamaparse_raw_result.json


In [11]:
job_info = {
    "job_id": result.job.id,
    "status": result.job.status,
    "file_id": uploaded_file.id,
    "source": str(PDF_PATH),
}

with (OUTPUT_DIR / "job_info.json").open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        job_info,
        file,
        indent=2,
        ensure_ascii=False
    )

In [12]:
full_markdown = result.markdown_full or ""

if not full_markdown and result.markdown:
    full_markdown = "\n\n---\n\n".join(
        getattr(page, "markdown", "")
        for page in result.markdown.pages
        if getattr(page, "success", True)
    )

full_text = result.text_full or ""

if not full_text and result.text:
    full_text = "\n\n".join(
        page.text
        for page in result.text.pages
    )


markdown_path = OUTPUT_DIR / "rag_ready_document.md"
text_path = OUTPUT_DIR / "extracted_text.txt"

markdown_path.write_text(
    full_markdown,
    encoding="utf-8"
)

text_path.write_text(
    full_text,
    encoding="utf-8"
)

print("Markdown saved:", markdown_path)
print("Plain text saved:", text_path)

Markdown saved: F:\Full_Stack_GenerativeAI_And_Agentic_AI\RAG\Data_parsing\data\llamaparse_output\rag_ready_document.md
Plain text saved: F:\Full_Stack_GenerativeAI_And_Agentic_AI\RAG\Data_parsing\data\llamaparse_output\extracted_text.txt


In [13]:
# ============================================================
# 8. Create Page-wise Records
# ============================================================

text_by_page = {}

if result.text:
    for page in result.text.pages:
        text_by_page[page.page_number] = page.text


metadata_by_page = {}

if result.metadata:
    for page in result.metadata.pages:
        metadata_by_page[page.page_number] = object_to_dict(page)


page_records = []

if result.markdown:
    for page in result.markdown.pages:
        page_number = page.page_number
        success = getattr(page, "success", True)

        if success:
            markdown_content = getattr(
                page,
                "markdown",
                ""
            )

            error_message = None
        else:
            markdown_content = ""
            error_message = getattr(
                page,
                "error",
                "Unknown parsing error"
            )


        page_records.append({
            "page_number": page_number,
            "success": success,
            "text": text_by_page.get(page_number, ""),
            "markdown": markdown_content,
            "header": getattr(page, "header", None),
            "footer": getattr(page, "footer", None),
            "error": error_message,
            "metadata": metadata_by_page.get(
                page_number,
                {}
            ),
        })

page_records_path = OUTPUT_DIR / "page_records.json"

with page_records_path.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        page_records,
        file,
        indent=2,
        ensure_ascii=False,
        default=str
    )

print("Page records saved:", page_records_path)
print("Total pages parsed:", len(page_records))

Page records saved: F:\Full_Stack_GenerativeAI_And_Agentic_AI\RAG\Data_parsing\data\llamaparse_output\page_records.json
Total pages parsed: 24
